<a href="https://colab.research.google.com/github/ithelga/beeline-banner-ab-test/blob/dev/notebooks/Team1_HW2_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SmartAds Efficiency**: Оптимизация эффективности маркетинговых каналов

## [STAGE 1] **Предобработка данных**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
import polars as pl
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

colors = ["#FFAFCC", "#FFC8DD", "#CDB4DB", "#BDE0FE", "#A2D2FF"] # СТАРЫЕ ЦВЕТА!!!!!!!!!!!!!!!!!!!!!!

graph_path = 'drive/MyDrive/Colab Notebooks/Beeline Banner AB-test/graph'
data_path = 'drive/MyDrive/Colab Notebooks/Beeline Banner AB-test/data'

Mounted at /content/drive


### Загрузка датасетов Google Disk

In [ ]:
CD_banner_df = pd.read_csv(f'{data_path}/CD_banner.csv', sep=";")
CD_banner_df.head(2)

,banner_id,creative_type (статика/видео/анимация),message (сообщение на баннере),size,target_audience_segment
0,1,video,Сервис,300x250,value
1,2,rich,Новинка,300x250,youth


In [ ]:
CD_campaign_df = pd.read_csv(f'{data_path}/CD_campaign.csv', sep=";")
CD_campaign_df.head(2)

,campaign_id,daily_budget,start_date,end_date
0,1,175071.43,01.02.2025,17.03.2025
1,2,95601.86,01.02.2025,17.03.2025


In [ ]:
CD_user_df = pd.read_csv(f'{data_path}/CD_user.csv', sep=";")
CD_user_df.head(2)

,User_id,segment,tariff,date_create,date_end
0,1,premium,premium,2025-01-18,2025-02-25
1,2,family,basic,2024-12-18,2025-01-20


In [ ]:
Fct_actions_df = pd.read_csv(f'{data_path}/Fct_actions.csv', sep=";")
Fct_actions_df.head(2)

,user_id,session_start,"actions (регистрация, первый заказ и т.д.)"
0,2908,2025-02-02 12:49:24,registration
1,2908,2025-02-03 10:11:53,first_order


In [ ]:
Fct_banners_show_df = pd.read_csv(f'{data_path}/Fct_banners_show.csv', sep=";")
Fct_banners_show_df.head(2)

,banner_id,campaign_id,user_id,timestamp,placement (сайт/приложение/соцсеть),device_type,os,geo,is_clicked (0/1)
0,1,1,38232,2025-02-01 05:54:06,app,phone,ios,Москва,0
1,1,1,58885,2025-02-01 22:30:50,site,phone,ios,Санкт-Петербург,0


In [ ]:
Installs_df = pd.read_csv(f'{data_path}/Installs.csv', sep=";")
Installs_df.head(2)

,user_id,install_timestamp,source (баннер / органика / другое)
0,2908,2025-02-02 10:56:24,banner
1,61135,2025-02-02 05:37:53,banner


### Предобработка данных

Получим описательную статистику и основные характеристики данных, чтобы понять, есть ли проблемные датасеты.

In [ ]:
dfs = {
    "CD_banner_df": CD_banner_df,
    "CD_campaign_df": CD_campaign_df,
    "CD_user_df": CD_user_df,
    "Fct_actions_df": Fct_actions_df,
    "Fct_banners_show_df": Fct_banners_show_df,
    "Installs_df": Installs_df
}

def quick_check(name, df):
    print(f"\n {name}")

    # Типы данных
    print("\n▶ Типы данных:")
    print(df.dtypes)

    # Пропуски
    print("\n▶ Пропуски:")
    print(df.isna().sum())

    # Дубликаты
    print("\n▶ Дубликаты:")
    print(df.duplicated().sum())

    # Описательные
    print("\n▶ Describe (коротко):")
    print(df.describe(include='all').T)


In [ ]:
for name, df in dfs.items():
    quick_check(name, df)


 CD_banner_df

▶ Типы данных:
banner_id                                  int64
creative_type (статика/видео/анимация)    object
message (сообщение на баннере)            object
size                                      object
target_audience_segment                   object
dtype: object

▶ Пропуски:
banner_id                                 0
creative_type (статика/видео/анимация)    0
message (сообщение на баннере)            0
size                                      0
target_audience_segment                   0
dtype: int64

▶ Дубликаты:
0

▶ Describe (коротко):
                                       count unique      top freq mean  \
banner_id                               15.0    NaN      NaN  NaN  8.0   
creative_type (статика/видео/анимация)    15      3   static    8  NaN   
message (сообщение на баннере)            15      6   Сервис    4  NaN   
size                                      15      3  300x250    8  NaN   
target_audience_segment                   15      3    

Видим, что самый проблемный это **Fct_banners_show** датасет. Аномальных значений не выявлено, категории выглядят чисто, пропуски в **CD_user_df** закономерная история, удаления не требуют, так что работаем только с **Fct_banners_show**


In [ ]:
# Ищем  идентичные строки
dups = Fct_banners_show_df[Fct_banners_show_df.duplicated(keep=False)]

# Собираем пары дубликатов
groups = dups.groupby(list(Fct_banners_show_df.columns))
pairs = []

for _, g in groups:
    if len(g) > 1:
        pairs.append(g.head(2))  # берем только пару, не весь блок
duplicate_pairs_df = pd.concat(pairs, axis=0)

duplicate_pairs_df.head(10)



,banner_id,campaign_id,user_id,timestamp,placement (сайт/приложение/соцсеть),device_type,os,geo,is_clicked (0/1)
1040240,1,1,157,2025-03-12 16:47:30,social,phone,ios,Екатеринбург,0
1200106,1,1,157,2025-03-12 16:47:30,social,phone,ios,Екатеринбург,0
159713,1,1,1838,2025-02-07 22:18:19,site,tablet,ios,Санкт-Петербург,0
1202311,1,1,1838,2025-02-07 22:18:19,site,tablet,ios,Санкт-Петербург,0
320710,1,1,2036,2025-02-13 13:58:46,app,phone,ios,Казань,0
1201750,1,1,2036,2025-02-13 13:58:46,app,phone,ios,Казань,0
83492,1,1,2508,2025-02-04 00:34:03,site,phone,ios,Москва,0
1201556,1,1,2508,2025-02-04 00:34:03,site,phone,ios,Москва,0
1147130,1,1,4099,2025-03-16 22:46:44,site,phone,ios,Казань,0
1200570,1,1,4099,2025-03-16 22:46:44,site,phone,ios,Казань,0


Решение - очистка от дубликатов.

In [ ]:
Fct_banners_show_df = Fct_banners_show_df.drop_duplicates().reset_index(drop=True)

Исследуем пропуски

In [ ]:
# Проверяем, нет ли сегментов, где пропуски встречаются устойчиво чаще
na_mask = Fct_banners_show_df.isna().any(axis=1)
cols_to_check = ["placement (сайт/приложение/соцсеть)", "device_type", "os", "geo"]

for col in cols_to_check:
    print(f"\n{col.upper()}")
    print(
        Fct_banners_show_df.assign(is_na=na_mask)
        .groupby(col)["is_na"]
        .mean()
        .sort_values(ascending=False)
        .round(4)
    )


PLACEMENT (САЙТ/ПРИЛОЖЕНИЕ/СОЦСЕТЬ)
placement (сайт/приложение/соцсеть)
site      0.0201
app       0.0198
social    0.0197
Name: is_na, dtype: float64

DEVICE_TYPE
device_type
phone     0.0199
tablet    0.0196
Name: is_na, dtype: float64

OS
os
ios        0.0300
android    0.0293
web        0.0290
Name: is_na, dtype: float64

GEO
geo
Омск               0.0204
Нижний Новгород    0.0203
Ростов-на-Дону     0.0199
Челябинск          0.0199
Самара             0.0199
Санкт-Петербург    0.0197
Новосибирск        0.0197
Москва             0.0197
Екатеринбург       0.0194
Казань             0.0194
Name: is_na, dtype: float64


Удаляем, поскольку сильных перекосов нет, а сам процент пропусков незначителен на таком объеме данных

In [ ]:
Fct_banners_show_df = Fct_banners_show_df.dropna()

На всякий случай, проверим ошибки в датах

In [ ]:
try:
    Fct_banners_show_df["timestamp"] = pd.to_datetime(Fct_banners_show_df["timestamp"], errors="coerce")
except Exception as e:
    print("Ошибка конвертации дат:", e)

print("\n🔍 Строки с некорректными датами:")
Fct_banners_show_df[Fct_banners_show_df["timestamp"].isna()]



🔍 Строки с некорректными датами:


,banner_id,campaign_id,user_id,timestamp,placement (сайт/приложение/соцсеть),device_type,os,geo,is_clicked (0/1)


Ошибок нет. Далее проверим качество формирования категорий

In [ ]:
for col in ["placement (сайт/приложение/соцсеть)", "device_type", "os", "geo"]:
    print(f"\n🔍 Уникальные в {col}:")
    print(Fct_banners_show_df[col].value_counts(dropna=False))



🔍 Уникальные в placement (сайт/приложение/соцсеть):
placement (сайт/приложение/соцсеть)
app       467144
site      464320
social    233128
Name: count, dtype: int64

🔍 Уникальные в device_type:
device_type
phone     990062
tablet    174530
Name: count, dtype: int64

🔍 Уникальные в os:
os
android    524592
ios        523697
web        116303
Name: count, dtype: int64

🔍 Уникальные в geo:
geo
Москва             349252
Санкт-Петербург    139493
Ростов-на-Дону     104843
Омск               104261
Новосибирск         93379
Казань              81553
Екатеринбург        81548
Нижний Новгород     70307
Челябинск           70134
Самара              69822
Name: count, dtype: int64


Аномалий не выявлено, оставляем. Далее приведем все даты к единому формату.

In [ ]:
date_columns = {
    "Fct_banners_show_df": ["timestamp"],
    "Installs_df": ["install_timestamp"],
    "Fct_actions_df": ["session_start"],
    "CD_campaign_df": ["start_date", "end_date"],
    "CD_user_df": ["date_create", "date_end"],
}


def convert_date_columns(df, df_name):
    cols = date_columns.get(df_name, [])
    print(f"\n {df_name}: обработка дат {cols}")

    for col in cols:
        if col not in df.columns:
            print(f"⚠ {col} отсутствует в датафрейме")
            continue

        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


In [ ]:
for name, df in dfs.items():
    print(f"\n {name}")

    # Применяем преобразование
    dfs[name] = convert_date_columns(df, name)

    # Какие колонки являются датами
    cols = date_columns.get(name, [])

    if not cols:
        print("Нет колонок-дат.")
        continue

    for col in cols:
        # пример значения — первое непустое
        example = df[col].dropna().iloc[0] if df[col].notna().any() else "нет данных"

        # формат (тип)
        dtype = df[col].dtype

        # процент пропусков
        na_pct = df[col].isna().mean().round(4)

        print(f"{col}: пример {example}, формат: {dtype}, пропуски = {na_pct}")




 CD_banner_df

 CD_banner_df: обработка дат []
Нет колонок-дат.

 CD_campaign_df

 CD_campaign_df: обработка дат ['start_date', 'end_date']
start_date: пример 2025-01-02 00:00:00, формат: datetime64[ns], пропуски = 0.0
end_date: пример 2025-03-17 00:00:00, формат: datetime64[ns], пропуски = 0.0

 CD_user_df

 CD_user_df: обработка дат ['date_create', 'date_end']
date_create: пример 2025-01-18 00:00:00, формат: datetime64[ns], пропуски = 0.0
date_end: пример 2025-02-25 00:00:00, формат: datetime64[ns], пропуски = 0.7723

 Fct_actions_df

 Fct_actions_df: обработка дат ['session_start']
session_start: пример 2025-02-02 12:49:24, формат: datetime64[ns], пропуски = 0.0

 Fct_banners_show_df

 Fct_banners_show_df: обработка дат ['timestamp']
timestamp: пример 2025-02-01 05:54:06, формат: datetime64[ns], пропуски = 0.0

 Installs_df

 Installs_df: обработка дат ['install_timestamp']
install_timestamp: пример 2025-02-02 10:56:24, формат: datetime64[ns], пропуски = 0.0


Последняя проверка перед объединением в витрину для анализа - проверка связей ID.

In [ ]:
CD_banner_df['banner_id'] = CD_banner_df['banner_id'].astype(str)
CD_campaign_df['campaign_id'] = CD_campaign_df['campaign_id'].astype(str)

Fct_banners_show_df['banner_id'] = Fct_banners_show_df['banner_id'].astype(str)
Fct_banners_show_df['campaign_id'] = Fct_banners_show_df['campaign_id'].astype(str)
Fct_banners_show_df['user_id'] = Fct_banners_show_df['user_id'].astype(str)

Installs_df['user_id'] = Installs_df['user_id'].astype(str)
Fct_actions_df['user_id'] = Fct_actions_df['user_id'].astype(str)
CD_user_df['User_id'] = CD_user_df['User_id'].astype(str)


Проверяем отсутствие ID по "материнским" датасетам.

In [ ]:
missing_banners = Fct_banners_show_df[~Fct_banners_show_df['banner_id']
                                      .isin(CD_banner_df['banner_id'])]['banner_id'].unique()



missing_campaigns = Fct_banners_show_df[~Fct_banners_show_df['campaign_id']
                                        .isin(CD_campaign_df['campaign_id'])]['campaign_id'].unique()

missing_users_show = Fct_banners_show_df[~Fct_banners_show_df['user_id']
                                         .isin(CD_user_df['User_id'])]['user_id'].unique()

missing_users_inst = Installs_df[~Installs_df['user_id']
                                 .isin(CD_user_df['User_id'])]['user_id'].unique()

missing_users_act = Fct_actions_df[~Fct_actions_df['user_id']
                                   .isin(CD_user_df['User_id'])]['user_id'].unique()




In [ ]:
print("missing banners:", len(missing_banners))
print("missing campaigns:", len(missing_campaigns))
print("missing users (show):", len(missing_users_show))
print("missing users (inst):", len(missing_users_inst))
print("missing users (actions):", len(missing_users_act))


missing banners: 0
missing campaigns: 0
missing users (show): 0
missing users (inst): 0
missing users (actions): 0


In [ ]:
# Упрощение названий

Fct_banners_show_df = Fct_banners_show_df.rename(columns={
    'is_clicked (0/1)': 'is_clicked',
    'placement (сайт/приложение/соцсеть)': 'placement'
})

CD_banner_df = CD_banner_df.rename(columns={
    'creative_type (статика/видео/анимация)': 'creative_type',
    'message (сообщение на баннере)': 'message'
})

Fct_actions_df = Fct_actions_df.rename(columns={
    'actions (регистрация, первый заказ и т.д.)': 'actions'
})

Installs_df = Installs_df.rename(columns={
    'source (баннер / органика / другое)': 'source'
})

Проблем не выявлено. Приступим к сбору витрины данных из доступных датасетов.

In [ ]:
# Убедимся, что в Fct_actions_df есть колонка date (день действия)
if "date" not in Fct_actions_df.columns:
    # если у тебя колонка session_start — используем её
    if "session_start" in Fct_actions_df.columns:
        Fct_actions_df["date"] = pd.to_datetime(Fct_actions_df["session_start"]).dt.floor("D")
    else:
        # если уже есть timestamp/другое имя — попытка авто
        Fct_actions_df["date"] = pd.to_datetime(Fct_actions_df.iloc[:,1]).dt.floor("D")

# 1) Собираем даты из всех источников
dates_shows   = Fct_banners_show_df[["user_id","timestamp"]].drop_duplicates()
dates_installs = install_events[["user_id","date"]].drop_duplicates()   # уже есть ранее
dates_actions_all = Fct_actions_df[["user_id","date"]].drop_duplicates()  # все действия (включая first_order/tariff_switch)

# объединяем всё — теперь включены даты first_order и tariff_switch
all_dates = (
    pd.concat([dates_shows, dates_installs, dates_actions_all], axis=0)
      .drop_duplicates()
      .sort_values(["user_id","date"])
      .reset_index(drop=True)
)

# 2) Мерджим shows_agg справа — чтобы пустые дни тоже присутствовали
tmp = all_dates.merge(shows_agg, on=["user_id","date"], how="left")

# 3) Мерджим install event (в конкретные дни) — если install_events ранее определён
tmp = tmp.merge(install_events, on=["user_id","date"], how="left")

# 4) Пересобираем action_events так, чтобы в нём были все нужные колонки:
#    (агрегируем count по (user_id,date,actions) и превращаем >0 -> 1)

tmp_actions = (
    Fct_actions_df
      .groupby(["user_id","date","actions"])
      .size()
      .reset_index()
)

tmp_actions = tmp_actions.rename(columns={0: "cnt"})  # старый pandas создаёт колонку 0

# Теперь pivot
action_pivot = (
    tmp_actions
      .pivot_table(
          index=["user_id","date"],
          columns="actions",
          values="cnt",
          fill_value=0
      )
      .reset_index()
)

# нормализуем имена колонок в формат has_*_event
col_map = {}
if "registration" in action_pivot.columns:
    col_map["registration"] = "has_registration_event"
if "first_order" in action_pivot.columns:
    col_map["first_order"] = "has_first_order_event"
if "tariff_switch" in action_pivot.columns:
    col_map["tariff_switch"] = "has_tariff_switch_event"

action_pivot = action_pivot.rename(columns=col_map)

# приводим значения к 0/1
for c in list(col_map.values()):
    action_pivot[c] = (action_pivot[c] > 0).astype(int)


# 5) Мерджим action_events (теперь с first_order и tariff_switch)
tmp = tmp.merge(action_pivot, on=["user_id","date"], how="left")

# 6) Мерджим user attributes (статические)
tmp = tmp.merge(user_attrs, on="user_id", how="left")

# 7) Заполняем пропуски по событиям
for c in [
    "shows","clicks",
    "has_install_event",
    "has_registration_event",
    "has_first_order_event",
    "has_tariff_switch_event"
]:
    if c in tmp.columns:
        tmp[c] = tmp[c].fillna(0).astype(int)

# 8) Add campaign / banner attributes (мэпим только если есть shows)
tmp["creative_type"] = tmp["banner_first"].map(lambda b: banner_map.get(b, {}).get("creative_type"))
tmp["size"] = tmp["banner_first"].map(lambda b: banner_map.get(b, {}).get("size"))
tmp["target_audience_segment"] = tmp["banner_first"].map(lambda b: banner_map.get(b, {}).get("target_audience_segment"))

# финально
supertable = tmp.sort_values(["user_id","date"]).reset_index(drop=True)
supertable["creative_type"] = supertable["creative_type"].fillna("no_impression")
supertable["campaign_first"] = supertable["campaign_first"].fillna("no_impression")
supertable["size"] = supertable["size"].fillna("no_impression")
supertable["target_audience_segment"] = supertable["target_audience_segment"].fillna("no_impression")



In [ ]:
# 1. Подготовим бюджет
camp_budget = (
    CD_campaign_df[["campaign_id", "daily_budget"]]
      .rename(columns={"campaign_id": "campaign_first"})
)

# 2. Группы
group_vars = ["date", "campaign_first"]

# 3. Агрегация факт-событий
campaign_funnel = (
    supertable
      .groupby(group_vars, as_index=False)
      .agg(
          shows_sum              = ("shows", "sum"),
          clicks_sum             = ("clicks", "sum"),
          installs_sum           = ("has_install_event", "sum"),
          registrations_sum      = ("has_registration_event", "sum"),
          first_order_sum        = ("has_first_order_event", "sum"),
          tariff_switch_sum      = ("has_tariff_switch_event", "sum"),
      )
)

# 4. Подцепляем бюджет
campaign_funnel = campaign_funnel.merge(
    camp_budget,
    on="campaign_first",
    how="left"
)

# 5. Метрики воронки

campaign_funnel["CTR"] = (
    campaign_funnel["clicks_sum"] / campaign_funnel["shows_sum"]
).fillna(0)

campaign_funnel["CR_install"] = (
    campaign_funnel["installs_sum"] / campaign_funnel["clicks_sum"]
).fillna(0)

campaign_funnel["CR_registration"] = (
    campaign_funnel["registrations_sum"] / campaign_funnel["installs_sum"]
).fillna(0)

campaign_funnel["CR_first_order"] = (
    campaign_funnel["first_order_sum"] / campaign_funnel["registrations_sum"]
).fillna(0)

campaign_funnel["CR_tariff_switch"] = (
    campaign_funnel["tariff_switch_sum"] / campaign_funnel["first_order_sum"]
).fillna(0)

# 6. Стоимость целевого действия (tariff_switch)
campaign_funnel["CPTS"] = (
    campaign_funnel["daily_budget"] / campaign_funnel["tariff_switch_sum"]
).replace([np.inf, np.nan], 0)


In [ ]:
campaign_funnel

,date,campaign_first,shows_sum,clicks_sum,installs_sum,registrations_sum,first_order_sum,tariff_switch_sum,daily_budget,CTR,CR_install,CR_registration,CR_first_order,CR_tariff_switch,CPTS
0,2025-02-01,1,3,0,3,2,0,0,175071.43,0.000000,inf,0.666667,0.000000,0.0,0.0
1,2025-02-01,2,7,2,6,6,0,0,95601.86,0.285714,3.0,1.000000,0.000000,0.0,0.0
2,2025-02-01,3,6,0,6,6,2,0,166617.61,0.000000,inf,1.000000,0.333333,0.0,0.0
3,2025-02-01,no_impression,0,0,18,17,1,0,NaN,0.000000,inf,0.944444,0.058824,0.0,0.0
4,2025-02-02,1,10,0,8,8,1,0,175071.43,0.000000,inf,1.000000,0.125000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025-04-02,no_impression,0,0,0,0,0,6,NaN,0.000000,0.0,0.000000,0.000000,inf,0.0
196,2025-04-03,no_impression,0,0,0,0,0,12,NaN,0.000000,0.0,0.000000,0.000000,inf,0.0
197,2025-04-04,no_impression,0,0,0,0,0,4,NaN,0.000000,0.0,0.000000,0.000000,inf,0.0
198,2025-04-05,no_impression,0,0,0,0,0,7,NaN,0.000000,0.0,0.000000,0.000000,inf,0.0


Проверяем, что ничего не потеряли

In [ ]:
print("Размер витрины:", supertable.shape)

# 1. Нет ли дублей по user_id + date
dupes = supertable.groupby(["user_id", "date"]).size().reset_index(name="cnt")
print("Дубликаты user_id+date:", dupes[dupes["cnt"] > 1].shape[0])

# 2. Нет ли строк без даты показа
print("Пустые даты показов:", supertable["date"].isna().sum())

# 3. Нет ли user_id NULL
print("Пустые user_id:", supertable["user_id"].isna().sum())

# 4. Проверим базовые распределения
print("\nПример дат:")
print(supertable["date"].value_counts().head())

print("\nКлики:")
print(supertable["clicks"].describe())



Размер витрины: (1171848, 20)
Дубликаты user_id+date: 0
Пустые даты показов: 1164590
Пустые user_id: 0

Пример дат:
date
2025-03-09    182
2025-02-27    179
2025-03-15    177
2025-02-23    173
2025-03-03    173
Name: count, dtype: int64

Клики:
count    1.171848e+06
mean     1.339764e-04
std      1.157405e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: clicks, dtype: float64


In [ ]:
campaign_funnel['installs_sum'].sum()

np.int64(3597)

In [ ]:
supertable['has_install_event'].sum()

np.int64(3597)

In [ ]:
# Базовая аналитика по витрине

analytics_daily = (
    supertable
    .groupby("date")
    .agg(
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_registration	= ("has_registration_event", "sum"),
        has_first_order	= ("has_first_order_event", "sum"),
        has_tariff_switch= ("has_tariff_switch_event", "sum")
    )
    .reset_index()
)

analytics_daily["ctr"] = analytics_daily["clicks"] / analytics_daily["shows"]

print("\n Дневные метрики")
print(analytics_daily.head())


# Разрез по гео
geo_stats = (
    supertable
    .groupby("geo")
    .agg(
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        users=("user_id", "nunique"),
        has_registration	= ("has_registration_event", "sum"),
        has_first_order	= ("has_first_order_event", "sum"),
        has_tariff_switch= ("has_tariff_switch_event", "sum")
    )
    .reset_index()
)

geo_stats["ctr"] = geo_stats["clicks"] / geo_stats["shows"]

print("\n Гео метрики")
print(geo_stats.sort_values("shows", ascending=False).head())


#  Разрез по устройствам
device_stats = (
    supertable
    .groupby("device_type")
    .agg(
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        users=("user_id", "nunique"),
        has_registration	= ("has_registration_event", "sum"),
        has_first_order	= ("has_first_order_event", "sum"),
        has_tariff_switch= ("has_tariff_switch_event", "sum")
    )
    .reset_index()
)

device_stats["ctr"] = device_stats["clicks"] / device_stats["shows"]

print("\n Device метрики")
print(device_stats)




 Дневные метрики
        date  shows  clicks  has_registration  has_first_order  \
0 2025-02-01     16       2                31                3   
1 2025-02-02     21       1                52               18   
2 2025-02-03     24       0                70               37   
3 2025-02-04     30       4                79               46   
4 2025-02-05     29       4                73               54   

   has_tariff_switch       ctr  
0                  0  0.125000  
1                  0  0.047619  
2                  0  0.000000  
3                  1  0.133333  
4                  5  0.137931  

 Гео метрики
               geo  shows  clicks  users  has_registration  has_first_order  \
2           Москва    522      52    436               241              157   
8  Санкт-Петербург    223      15    195                99               75   
5             Омск    181      17    145                77               60   
6   Ростов-на-Дону    167      16    153                7

### Сохранение данных на Google Disk

In [ ]:
supertable.to_csv(f'{data_path}/supertable.csv', index=False)

In [ ]:
campaign_funnel.to_csv(f'{data_path}/campaign_funnel.csv', index=False)